# 06 — 最終課題：手法を「選べる」状態へ
ここまでの一連の変化を式と挙動の両方で比較する。

| 手法 | 学習方法 | 主な効果 |
|---|---|---|
| CART | 1本の木を貪欲分割 | 解釈しやすいが高分散 |
| Bagging / Random Forest | 木を並列に訓練して平均 | 分散低減、相関低減 |
| Gradient Boosting | 負の勾配に木を逐次当てる | 加法的に誤差を補正 |
| XGBoost | 2次近似の改善量で木を逐次追加 | 曲率・正則化・効率化 |

**最終提出物の条件:** 学習アルゴリズムの違い、評価設計、ハイパーパラメータの影響、失敗例を一つずつ説明できること。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_friedman1
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
X,y=make_friedman1(n_samples=800,n_features=10,noise=1.0,random_state=42)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,random_state=42)
models={
    "Tree":DecisionTreeRegressor(max_depth=5,random_state=42),
    "Random Forest":RandomForestRegressor(n_estimators=100,max_depth=5,n_jobs=2,random_state=42),
    "Gradient Boosting":GradientBoostingRegressor(n_estimators=100,max_depth=3,learning_rate=.1,random_state=42),
    "XGBoost":XGBRegressor(n_estimators=100,max_depth=3,learning_rate=.1,
                         tree_method="hist",n_jobs=2,random_state=42),
}
scores={}
for name,m in models.items():
    m.fit(X_train,y_train)
    scores[name]=mean_squared_error(y_test,m.predict(X_test))
print(scores)
plt.barh(list(scores),list(scores.values()))
plt.xlabel("Test MSE (lower is better)")
plt.title("Single seeded split; for illustration, not a ranking")
plt.show()

## 発展：単一の test split でランキングしてはいけない
上の図は学習用の観察であり、一般的な優劣を証明しない。異なる train/test split や交差検証でどれだけ順位が変わるかを確かめよう。**パラメータ選定**には交差検証、最後の性能報告には独立したテストを用いる。

実験テンプレート:
1. train/validation を分離し、パラメータだけを validation または CV で選ぶ
2. 指標を先に決める（MSE、MAE、AUC、logloss など）
3. 変更した条件・ランダム seed・失敗例を記録する
4. 最後の test は1回だけ評価する
5. 精度以外に学習時間・推論時間・解釈性も比較する

**考察問い:** (a) 小標本で深い木が危険な理由は？ (b) 木同士の相関が高いと bagging はどうなる？ (c) XGBoost の $\gamma$ と $\lambda$ は異なる何を罰する？ (d) テストデータで early stopping してはいけない理由は？

In [ ]:
# 採点を全件通したら、説明問題を README にまとめて自分の言葉で答える。
print("最終確認: python tools/grade.py")
print("答え合わせ: solutions/reference.py を最後に読む")